# Workshop 1.5: Merging and Joining DataFrames

Welcome to Workshop 1.5! In institutional quantitative research, market information rarely lives inside a single tidy table. Instead, price histories, corporate earnings, and macroeconomic indicators arrive from different feeds and databases.

### Unifying Disparate Data Sources

To test multi-factor models or analyze fundamental signals, we need to stitch disparate tables together. If we want to evaluate price momentum alongside quarterly financial health, we must join our price table with corporate earnings.

Pandas equips us with two essential joining mechanisms:
- **`pd.merge()`**: Connects tables based on exact matching keys, functioning like relational SQL joins.
- **`pd.merge_asof()`**: Aligns non-synchronous time series, matching each observation with the most recent event known at that moment.

In this workshop, we will master relational joins and learn how `pd.merge_asof()` protects our strategy simulations from subtle look-ahead bias.

> **Key Takeaway**: Combining tables with `pd.merge()` and `pd.merge_asof()` enables us to blend diverse market datasets into unified research tables.

## Topic 1: Standard pd.merge(): The Relational Join

Think of **`pd.merge()`** like matching puzzle pieces. When two tables share a common identifier, such as a customer ID or ticker symbol, pandas pairs corresponding rows together into a unified row.

Let's create two related tables: a customer registry and a table of purchase orders. Let's see:

In [1]:
import pandas as pd

# Table 1: Customers
customers = pd.DataFrame({
  "Customer_ID": [1, 2, 3],
  "Name": ["Alice", "Bob", "Charlie"]
})

# Table 2: Orders placed by customers:
orders = pd.DataFrame({
  "Order_ID": [101, 102, 103, 104],
  "Customer_ID": [1, 2, 1, 4], # Note: Customer 4 is not in the customers table!
  "Amount": [250, 150, 300, 400]
})

print("--- Customers Table ---")
print(customers)
print("\n--- Orders Table ---")
print(orders)

--- Customers Table ---
   Customer_ID     Name
0            1    Alice
1            2      Bob
2            3  Charlie

--- Orders Table ---
   Order_ID  Customer_ID  Amount
0       101            1      50
1       102            2     200
2       103            1      75


### Merging the Tables on Customer_ID

To fuse these tables, we supply both DataFrames and identify their shared key using `on="Customer_ID"`. By default, pandas carries out an **inner join**, keeping only rows where the key appears in both tables.

Let's merge our customer and order tables. Let's see:

In [2]:
# Perform an inner join on the shared 'Customer_ID' column:
merged_df = pd.merge(customers, orders, on="Customer_ID")

print(merged_df)

   Customer_ID     Name  Order_ID  Amount
0            1    Alice     101.0    50.0
1            1    Alice     103.0    75.0
2            2      Bob     102.0   200.0
3            3  Charlie       NaN     NaN


### Understanding Inner Join Outcomes

Notice how pandas matched the records:
- Customer 1 (`Alice`) placed two separate orders, so her name appears on two rows.
- Customer 3 (`Charlie`) placed no orders, so he is excluded from an inner join.
- Order 104 belongs to Customer 4, who does not exist in our customer list, so that order drops out.

> **Key Takeaway**: An inner join preserves only rows whose keys exist in both source tables.

## Topic 2: Controlling Join Types with how

Real data questions require different join strategies depending on which records you need to preserve. We control this behavior using the `how` argument:
- **`how="inner"`**: Keeps only matching rows found in both tables.
- **`how="left"`**: Preserves every row from the left table, inserting `NaN` when no matching record exists on the right.
- **`how="right"`**: Preserves every row from the right table.
- **`how="outer"`**: Keeps all rows from both tables, filling missing fields with `NaN`.

Let's compare left and outer joins directly. Let's see:

In [3]:
# Left Join: Keeps all customers, even those with no orders:
left_result = pd.merge(customers, orders, on="Customer_ID", how="left")
print("--- Left Join (Keeps all customers) ---")
print(left_result)

# Outer Join: Keeps all rows from both tables:
outer_result = pd.merge(customers, orders, on="Customer_ID", how="outer")
print("\n--- Outer Join (Keeps everyone) ---")
print(outer_result)

--- Inner Join (Charlie is deleted!) ---
   Customer_ID   Name  Order_ID  Amount
0            1  Alice       101      50
1            1  Alice       103      75
2            2    Bob       102     200

--- Outer Join (Keeps everyone) ---
   Customer_ID     Name  Order_ID  Amount
0            1    Alice     101.0    50.0
1            1    Alice     103.0    75.0
2            2      Bob     102.0   200.0
3            3  Charlie       NaN     NaN


> **Key Takeaway**: Use `how="left"` when you must preserve all primary entities, and `how="outer"` to capture every available observation across both datasets.

---

## Topic 3: Merging with Mismatched Column Headers

In practice, databases managed by separate teams often assign different labels to the same underlying entity. For example, one table might label an identifier as `"ID"` while another calls it `"Customer_ID"`.

Rather than manually renaming columns first, pandas lets us specify `left_on` and `right_on` keys explicitly.

Let's see how pandas bridges mismatched column headers. Let's check:

In [4]:
# Suppose the customer table named its column 'ID' instead of 'Customer_ID':
customers_renamed = pd.DataFrame({
  "ID": [1, 2, 3],
  "Name": ["Alice", "Bob", "Charlie"]
})

# Merge using left_on and right_on:
diff_merge = pd.merge(customers_renamed, orders, left_on="ID", right_on="Customer_ID", how="left")

print(diff_merge)

   ID     Name  Order_ID  Customer_ID  Amount
0   1    Alice     101.0            1    50.0
1   1    Alice     103.0            1    75.0
2   2      Bob     102.0            2   200.0
3   3  Charlie       NaN          NaN     NaN


> **Key Takeaway**: Use `left_on` and `right_on` to connect tables when column names differ across datasets.

---

## Topic 4: Aligning Non-Synchronous Timestamps with pd.merge_asof()

Standard `pd.merge()` requires exact key equality. But in financial markets, datasets are almost always **non-synchronous**:
- Stock prices arrive continuously every single trading day.
- Corporate earnings, central bank announcements, or economic indicators arrive irregularly.

If we attempt an exact merge on dates, days without announcements will fail to match! To solve this, pandas provides **`pd.merge_asof()`**, which stands for *merge as of this date*.

For each daily price observation, `pd.merge_asof()` identifies the most recent event that occurred on or before that date.

Let's create daily prices and irregular corporate announcements. Let's see:

In [5]:
# Daily stock prices:
prices = pd.DataFrame({
  "Date": pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04"]),
  "Price": [100, 101, 102, 103]
})

# Irregular corporate announcements (only happened on Jan 1 and Jan 3):
events = pd.DataFrame({
  "Event_Date": pd.to_datetime(["2024-01-01", "2024-01-03"]),
  "Event": ["Announcement A", "Announcement B"]
})

print("--- Daily Prices ---")
print(prices)
print("\n--- Irregular Events ---")
print(events)

--- Daily Prices ---
        Date  Price
0 2024-01-01    100
1 2024-01-02    101
2 2024-01-03    102
3 2024-01-04    103

--- Irregular Events ---
  Event_Date           Event
0 2024-01-01  Announcement A
1 2024-01-03  Announcement B


Now let's align them using `pd.merge_asof(..., direction="backward")`. Setting `direction="backward"` instructs pandas to look backward in time for the most recent known event.

Let's see how the backward alignment pairs our records. Let's check:

In [6]:
# Align events to daily prices looking backward in time:
aligned_backward = pd.merge_asof(
  prices,
  events,
  left_on="Date",
  right_on="Event_Date",
  direction="backward"
)

print(aligned_backward)

        Date  Price Event_Date           Event
0 2024-01-01    100 2024-01-01  Announcement A
1 2024-01-02    101 2024-01-01  Announcement A
2 2024-01-03    102 2024-01-03  Announcement B
3 2024-01-04    103 2024-01-03  Announcement B


Notice row 1 on `2024-01-02`: even though no announcement occurred on January 2, pandas accurately matches it with `Announcement A` because that was the latest known announcement as of that date.

> **Key Takeaway**: `pd.merge_asof(direction="backward")` matches each market date with the most recent prior event, ensuring our data alignment reflects real time.

## Topic 5: The Danger of direction="forward"

What happens if someone accidentally specifies `direction="forward"`? Instead of looking backward at historical facts, pandas searches forward into the future for upcoming announcements.

Let's observe what this forward lookup produces. Let's check:

In [7]:
# Dangerous: Looking forward into the future!
aligned_forward = pd.merge_asof(
  prices,
  events,
  left_on="Date",
  right_on="Event_Date",
  direction="forward"
)

print(aligned_forward)

        Date  Price Event_Date           Event
0 2024-01-01    100 2024-01-01  Announcement A
1 2024-01-02    101 2024-01-03  Announcement B
2 2024-01-03    102 2024-01-03  Announcement B
3 2024-01-04    103        NaT             NaN


### Why Forward Merging Corrupts Backtests

Inspect row 1 on `2024-01-02`: it pairs January 2 with `Announcement B`, which does not occur until January 3. In live trading, nobody knows what will be announced tomorrow.

If our strategy enters positions based on future announcements, our backtest generates illusionary profits that fail in live markets.

> **Key Takeaway**: In quantitative backtesting, always enforce `direction="backward"` to prevent accidental future data leakage.

---

## Practice Time

Now it is your turn to practice relational merging and temporal alignment. Merging disparate tables is a daily task in quantitative analysis, so work through these challenges step by step.

---

### Challenge 1: Merging Products and Transactions

- Create a products DataFrame:
  ```python
  products = pd.DataFrame({
      "Product_ID": [10, 20, 30],
      "Name": ["Laptop", "Monitor", "Keyboard"]
  })
  ```
- Create a sales DataFrame:
  ```python
  sales = pd.DataFrame({
      "Sale_ID": [1, 2, 3],
      "Product_ID": [10, 20, 10],
      "Amount": [1200, 300, 1200]
  })
  ```
- Merge them using `pd.merge()` with a **left join** on `"Product_ID"` and display the result.

In [ ]:
# Challenge 1: Write your code below this line


# Expected Output:
#  Product_ID   Name Sale_ID Amount
# 0     10  Laptop   1.0 1200.0
# 1     10  Laptop   3.0 1200.0
# 2     20  Monitor   2.0  300.0
# 3     30 Keyboard   NaN   NaN


### Challenge 2: As-Of Merging on Macro Events

- Given daily weather records and irregular storm warnings:
  ```python
  weather = pd.DataFrame({
      "Date": pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
      "Temp": [22, 24, 19, 21, 23]
  })
  alerts = pd.DataFrame({
      "Alert_Date": pd.to_datetime(["2024-01-02", "2024-01-04"]),
      "Warning": ["High Wind", "Heavy Rain"]
  })
  ```
- Use `pd.merge_asof()` with `direction="backward"` to match warnings with daily weather dates.
- Display the aligned table.

In [ ]:
# Challenge 2: Write your code below this line


# Expected Output:
#     Date Temp Alert_Date   Warning
# 0 2024-01-01  22    NaT     NaN
# 1 2024-01-02  24 2024-01-02  High Wind
# 2 2024-01-03  19 2024-01-02  High Wind
# 3 2024-01-04  21 2024-01-04 Heavy Rain
# 4 2024-01-05  23 2024-01-04 Heavy Rain


### Challenge 3: Explaining Look-Ahead Bias

- In your own words, explain why using `direction="forward"` invalidates historical trading simulations.

In [ ]:
# Challenge 3: Write your explanation as a Python comment below:
#
#


---

## Solutions Section

Great work completing these merging exercises! Aligning asynchronous market streams cleanly is essential for building robust quantitative systems.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
products = pd.DataFrame({
    "Product_ID": [10, 20, 30],
    "Name": ["Laptop", "Monitor", "Keyboard"]
})
sales = pd.DataFrame({
    "Sale_ID": [1, 2, 3],
    "Product_ID": [10, 20, 10],
    "Amount": [1200, 300, 1200]
})
result1 = pd.merge(products, sales, on="Product_ID", how="left")
print(result1)
```

#### Solution for Challenge 2:
```python
weather = pd.DataFrame({
    "Date": pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
    "Temp": [22, 24, 19, 21, 23]
})
alerts = pd.DataFrame({
    "Alert_Date": pd.to_datetime(["2024-01-02", "2024-01-04"]),
    "Warning": ["High Wind", "Heavy Rain"]
})
result2 = pd.merge_asof(weather, alerts, left_on="Date", right_on="Alert_Date", direction="backward")
print(result2)
```

#### Solution for Challenge 3:
```python
# direction="forward" is dangerous because it looks into future rows that have not happened yet.
# In trading, this causes 'look-ahead bias', making simulated trades based on tomorrow's news
# before it actually occurs. It creates unrealistic backtest profits that will fail in live trading.
print("direction='forward' causes look-ahead bias by peeking into future data!")
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [8]:
# Solution for Challenge 1:
products = pd.DataFrame({
  "Product_ID": [10, 20, 30],
  "Name": ["Laptop", "Monitor", "Keyboard"]
})
sales = pd.DataFrame({
  "Sale_ID": [1, 2, 3],
  "Product_ID": [10, 20, 10],
  "Amount": [1200, 300, 1200]
})
result1 = pd.merge(products, sales, on="Product_ID", how="left")
print(result1)

   Product_ID      Name  Sale_ID  Amount
0          10    Laptop      1.0  1200.0
1          10    Laptop      3.0  1200.0
2          20   Monitor      2.0   300.0
3          30  Keyboard      NaN     NaN


In [9]:
# Solution for Challenge 2:
weather = pd.DataFrame({
  "Date": pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
  "Temp": [22, 24, 19, 21, 23]
})
alerts = pd.DataFrame({
  "Alert_Date": pd.to_datetime(["2024-01-02", "2024-01-04"]),
  "Warning": ["High Wind", "Heavy Rain"]
})
result2 = pd.merge_asof(weather, alerts, left_on="Date", right_on="Alert_Date", direction="backward")
print(result2)

        Date  Temp Alert_Date     Warning
0 2024-01-01    22        NaT         NaN
1 2024-01-02    24 2024-01-02   High Wind
2 2024-01-03    19 2024-01-02   High Wind
3 2024-01-04    21 2024-01-04  Heavy Rain
4 2024-01-05    23 2024-01-04  Heavy Rain


In [10]:
# Solution for Challenge 3:
# direction="forward" matches past dates with events that haven't occurred yet.
# This gives the strategy 'future vision', producing fake profits that cannot work in real life.
print("direction='forward' causes look-ahead bias by peeking into future data!")

direction='forward' causes look-ahead bias by peeking into future data!
